In [1]:
# Packages pypowsybl
import pypowsybl as pp
import pypowsybl.network as pn
import pypowsybl.loadflow as lf
# Detail logging packages (knitro)
import logging
import math
import pypowsybl.loadflow as lf
import json

# vizualization packages 
from slack_viz_utils import nad_explorer_with_slack,compute_slack_info, calculate_dc_losses
import pandas as pd


KNITRO SOLVER, with Pypowsybl

Input parameter is a network that does not converges with Newton-Raphson Method 

In [2]:
# Load the non convergent network 
Path_to_network = r"Outputs\ieee14-voltage-perturbation.xiidm"
network_KN = pn.load(Path_to_network)
network_NR = pn.load(Path_to_network)
network_DC = pn.load(Path_to_network)

In [3]:
# Newton Raphson solver diverges 
p_NR = lf.Parameters(provider_parameters={'acsolverType': 'NEWTON_RAPHSON'})
lf.run_ac(network_NR, p_NR )

[ComponentResult(connected_component_num=0, synchronous_component_num=0, status=MAX_ITERATION_REACHED, status_text=Reached Newton-Raphson max iterations limit, iteration_count=16, reference_bus_id='VL1_0', slack_bus_results=[SlackBusResult(id='VL1_0', active_power_mismatch=935142.9301484277)], distributed_active_power=0.0)]

In [4]:
# Run a DC LF on the perturbated network to approximate losses, and for a weight for the objective function 
lf.run_dc(network_DC)
voltage_level_df = network_DC.get_voltage_levels()
losses = calculate_dc_losses(network_DC, voltage_level_df)

print(f"DC LOSSES Approximation : {losses:.2f} MW")

DC LOSSES Approximation : 5.45 MW


In [5]:
# Parameters to choose the different level of logging details : DEBUG, INFO 
logging.basicConfig()
logging.getLogger('powsybl').setLevel(logging.DEBUG)

In [6]:
Path_to_CSV = r'TEST1_Result\ieee14-perturbation'
p_KN = lf.Parameters( distributed_slack=False, use_reactive_limits=False, provider_parameters={
  'acSolverType': 'KNITRO', 'solverType':'RELAXED', 'losses': str(losses), 
  'maxKnitroIterations': '200', 'gradientComputationMode': '1', 'threadNumber':'1', 
  'gradientUserRoutine': '2', 'hessianComputationMode': '6', 'minRealisticVoltage': '0.5', 
  'maxRealisticVoltage': '1.5', 'slackThreshold':'0.000001', 
  'relativeFeasibilityStoppingCriteria': '0.000001', 'absoluteFeasibilityStoppingCriteria':'0.001',
  'relativeOptimalityStoppingCriteria': '0.000001', 'absoluteOptimalityStoppingCriteria': ' 0.001', 'optimalityStoppingCriteria':'0.0000001', 
  'alwaysUpdateNetwork': 'false','exportSolution': Path_to_CSV })
# 'exportSolution': Path_to_CSV
lf.run_ac(network_KN, p_KN)

INFO:powsybl:loadflow provider used is : OpenLoadFlow
INFO:powsybl:Version: {repositoryName=powsybl-open-loadflow, mavenProjectVersion=2.1.1-not-vectorized, gitVersion=72a7012b7d75726be512be08328e097453f67b2a, gitBranch=olf-2.1.1-not-vectorized, buildTimestamp=2026-06-16T13:01:47.882Z}
INFO:powsybl:Parameters:
┌───────────────────────────┬───────────────────────────────────────────────────────────┬──────────────────────────────────┐
│ Category                  │ Name                                                      │ Value                            │
├───────────────────────────┼───────────────────────────────────────────────────────────┼──────────────────────────────────┤
│ Automation                │ simulateAutomationSystems                                 │ false                            │
│ DC                        │ dcApproximationType                                       │ IGNORE_R                         │
│                           │ dcPowerFactor                    

[ComponentResult(connected_component_num=0, synchronous_component_num=0, status=CONVERGED, status_text=Converged, iteration_count=12, reference_bus_id='VL1_0', slack_bus_results=[SlackBusResult(id='VL1_0', active_power_mismatch=293.8512900970478)], distributed_active_power=0.0)]

Load Data of the slack information 

In [7]:
data = pd.read_csv(Path_to_CSV + ".csv", sep=";")
data

,bus_id,type,slackValue_pu,voltage_level_id,generator,controleVoltage,transfo,shunt,load,load_violation,gen_violation,outerloop_iteration
0,VL5_0,V,0.052949,VL5,NaN,"VoltageControl(type=GENERATOR, controlledBus='...",NaN,NaN,VL5_0_load,0,0,0


In [9]:
from pypowsybl.network import NadParameters

explorer = nad_explorer_with_slack(network_KN, slack_info=data, outerloop=0)
explorer

INFO:powsybl:Loading component library from /DefaultLibrary...
DEBUG:powsybl:Reading subComponent /DefaultLibrary/load.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/battery.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/generator.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/capacitor.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/inductor.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/svc.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/unknown-component.svg
INFO:powsybl:Loading component library from /DefaultLibrary...
DEBUG:powsybl:Reading subComponent /DefaultLibrary/load.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/battery.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/generator.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/capacitor.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/inductor.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrary/svc.svg
DEBUG:powsybl:Reading subComponent /DefaultLibrar